# Task 2 — Temporal Analysis of Groupings

Goal: 

- Membership change over time: do countries keep similar neighbors (i.e., remain in the same “visual group”), or do they drift to different neighborhoods?

- Defining-attributes change over time: which indicators most strongly align with the visual grouping at each year, and how does that mix evolve?

Key design choices (aligned with professor’s guidance):

- Still no imputation; we compute NaN-aware distances and build a yearly UMAP layout (like Task 1).

- We avoid hard clustering; instead we quantify neighborhood stability (Jaccard overlap of k-nearest neighbors year-to-year), which is visual-evidence friendly and does not “let an algorithm decide the groups.”

- For defining-attributes, we compute per year the axis-alignment score of each indicator with the 2D layout (absolute Spearman correlation with x and y, combined as a magnitude), and visualize this as time series / heatmaps.

Imports

In [1]:
# If running in a fresh environment, uncomment:
# !pip install -q pandas numpy plotly scikit-learn umap-learn tqdm

import os, math, json
import numpy as np
import pandas as pd

from tqdm import tqdm
import plotly.express as px
import plotly.graph_objects as go

from sklearn.manifold import TSNE
import umap

import warnings
warnings.filterwarnings("ignore")


C:\Users\shyam\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Load dataset & reuse Task-1 helpers

Adjust the DATA_PATH if needed. This cell also re-defines the NaN-aware z-score, NaN-aware distances, and the yearly embedding function used in Task 1.

In [2]:
# --- Load ---
DATA_PATH = r"Preprocessed-Data\WDI_cleaned_1975_2023.csv"
if not os.path.exists(DATA_PATH):
    alt = os.path.join(".", "Preprocessed-Data", "WDI_cleaned_1975_2023.csv")
    if os.path.exists(alt): DATA_PATH = alt

df = pd.read_csv(DATA_PATH)

# Standardise name variants
df.rename(columns={
    "Country Name": "Country",
    "Country_Name": "Country",
    "Country_Code": "Country Code",
}, inplace=True)

# Identify columns
years_all = sorted(df["Year"].dropna().unique().astype(int).tolist())
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
indicators = [c for c in numeric_cols if c.lower() != "year"]

def zscore_nan(series: pd.Series):
    mu = series.mean(skipna=True)
    sd = series.std(skipna=True)
    return (series - mu) / sd if (sd and not math.isclose(sd, 0.0)) else series - mu

norm_df = df.copy()
for c in indicators:
    norm_df[c] = zscore_nan(norm_df[c])

def naneuclid_matrix(X: np.ndarray) -> np.ndarray:
    n = X.shape[0]
    D = np.zeros((n, n), dtype=float)
    LARGE = 1e3
    for i in range(n):
        xi = X[i]
        for j in range(i+1, n):
            xj = X[j]
            mask = ~np.isnan(xi) & ~np.isnan(xj)
            if not np.any(mask):
                d = LARGE
            else:
                d = np.linalg.norm(xi[mask] - xj[mask])
            D[i, j] = D[j, i] = d
    return D

def build_year_embedding(year, method="umap", n_neighbors=15, min_dist=0.15, perplexity=30, random_state=42):
    ydf = norm_df[norm_df["Year"] == year].copy()
    if ydf.empty:
        raise ValueError(f"No rows for Year={year}")
    X = ydf[indicators].to_numpy(dtype=float)
    D = naneuclid_matrix(X)
    if method == "umap":
        reducer = umap.UMAP(
            n_neighbors=n_neighbors, min_dist=min_dist,
            metric="precomputed", random_state=random_state
        )
        coords = reducer.fit_transform(D)
        algo = "UMAP"
    elif method == "tsne":
        tsne = TSNE(
            n_components=2, metric="precomputed",
            perplexity=max(5, min(perplexity, max(5, D.shape[0]//3))),
            init="random", learning_rate="auto", random_state=random_state
        )
        coords = tsne.fit_transform(D)
        algo = "t-SNE"
    else:
        raise ValueError("method must be 'umap' or 'tsne'")
    out = ydf[["Country", "Country Code", "Year"] + indicators].copy()
    out["x"], out["y"], out["_algo"] = coords[:,0], coords[:,1], algo
    return out


# 2. Compute embeddings for all years

This is the heavy step. We’ll store results in a dict and a concatenated DataFrame for easy plotting/animation.

In [3]:
all_embeds = {}
frames = []
for y in tqdm(years_all, desc="Embedding per year (UMAP)"):
    try:
        emb = build_year_embedding(y, method="umap", n_neighbors=15, min_dist=0.15, random_state=42)
        all_embeds[y] = emb
        frames.append(emb)
    except Exception as e:
        print(f"Year {y} skipped due to: {e}")

embeds_df = pd.concat(frames, ignore_index=True)
len(all_embeds), embeds_df.shape, embeds_df["Year"].min(), embeds_df["Year"].max()


Embedding per year (UMAP): 100%|██████████| 49/49 [00:27<00:00,  1.81it/s]


(49, (11929, 14), np.int64(1975), np.int64(2023))

# 3. Animated temporal map (country trajectories + time slider)



- Shows movement of each country through the similarity space across years (membership drift).

- You can filter by region/country and see paths.

In [25]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Optional: bring in region if your CSV has it (else skip/filter by country only).
has_region = "Region" in embeds_df.columns

# 1️⃣ Build “trail” lines by sorting per country over time (static context layer)
traj = embeds_df.sort_values(["Country", "Year"]).copy()

# Create the base figure (context first)
fig_anim = go.Figure()

# Very faint, thin trajectories for historical motion context
for _, g in traj.groupby("Country", sort=False):
    fig_anim.add_trace(go.Scatter(
        x=g["x"], y=g["y"], mode="lines",
        line=dict(width=0.5, color="rgba(120,120,120,0.08)"),
        hoverinfo="skip", showlegend=False
    ))

# 2️⃣ Add animated scatter points (focus layer)
scatter = px.scatter(
    embeds_df, x="x", y="y",
    animation_frame="Year", animation_group="Country",
    hover_data=["Country"] + indicators[:10],
    color=indicators[0] if indicators else None,
    title="Temporal UMAP Map — Countries move through similarity space over time",
    height=750
)

# Copy animation data + frames from px.scatter
fig_anim.add_traces(scatter.data)
fig_anim.frames = scatter.frames

# ✅ Restore interactivity (slider + play/pause controls)
fig_anim.layout.updatemenus = scatter.layout.updatemenus
fig_anim.layout.sliders = scatter.layout.sliders

# 3️⃣ Highlight current-year scatter points (clarity focus)
fig_anim.update_traces(
    selector=dict(mode="markers"),
    marker=dict(size=7, opacity=0.95, line=dict(width=0.5, color="rgba(0,0,0,0.35)"))
)

# Apply styling to every animation frame
for fr in fig_anim.frames:
    for tr in fr.data:
        if getattr(tr, "mode", None) == "markers":
            tr.update(marker=dict(size=7, opacity=0.95,
                                  line=dict(width=0.5, color="rgba(0,0,0,0.35)")))

# 4️⃣ Layout polish
fig_anim.update_layout(
    xaxis_title="Similarity axis 1",
    yaxis_title="Similarity axis 2",
    margin=dict(l=10, r=10, t=60, b=10),
    legend_title_text=(indicators[0] if indicators else "Legend")
)

# 5️⃣ Render interactive animation
pio.renderers.default = "browser"
fig_anim.show()

# Save as fully interactive HTML (offline-compatible)
fig_anim.write_html(
    "figures/task2_temporal_umap.html",
    include_plotlyjs="embed",   # ensures offline interactivity
    full_html=True,
    auto_play=False
)

print("✅ Interactive Temporal UMAP successfully generated and saved to 'figures/task2_temporal_umap.html'")


✅ Interactive Temporal UMAP successfully generated and saved to 'figures/task2_temporal_umap.html'


How to read:

- Countries that stay with similar neighbors trace short, compact paths; those that change groups show long shifts or moves toward different neighborhoods.

- Use the player to scrub the Year. Hover any point for exact indicator values that year.

# 4. Neighborhood-stability metric (membership change without clustering)

We measure, for each country, how similar its neighborhood is from year→year using k-nearest neighbors (kNN) within each year’s embedding and computing the Jaccard overlap of neighbor sets. Low overlap ⇒ membership change; high overlap ⇒ stable membership.

In [5]:
from sklearn.neighbors import NearestNeighbors

def knn_neighbors(df_year, k=12):
    """Return dict: country -> set of neighbor country names (excluding self)."""
    XY = df_year[["x","y"]].to_numpy()
    nbrs = NearestNeighbors(n_neighbors=min(k+1, len(df_year))).fit(XY)
    _, idxs = nbrs.kneighbors(XY)
    neighbors = {}
    names = df_year["Country"].tolist()
    for i, name in enumerate(names):
        # exclude self at idx 0
        neigh = [names[j] for j in idxs[i] if j != i]
        neighbors[name] = set(neigh)
    return neighbors

def jaccard(a, b):
    if not a and not b: return 1.0
    if not a or not b: return 0.0
    inter = len(a & b)
    union = len(a | b)
    return inter/union if union > 0 else 0.0

# Compute per-country neighbor stability between consecutive years
k = 12
rows = []
for i in range(1, len(years_all)):
    y_prev, y_curr = years_all[i-1], years_all[i]
    if y_prev not in all_embeds or y_curr not in all_embeds: 
        continue
    A = all_embeds[y_prev][["Country","x","y"]].copy()
    B = all_embeds[y_curr][["Country","x","y"]].copy()
    # Align by countries that exist in both years
    common = set(A["Country"]).intersection(set(B["Country"]))
    A = A[A["Country"].isin(common)].reset_index(drop=True)
    B = B[B["Country"].isin(common)].reset_index(drop=True)
    neighA = knn_neighbors(A, k=k)
    neighB = knn_neighbors(B, k=k)
    for c in common:
        score = jaccard(neighA[c], neighB[c])
        rows.append({"Country": c, "Year_prev": y_prev, "Year": y_curr, "k": k, "neighbor_stability": score})

stability = pd.DataFrame(rows)
stability.head(), stability["neighbor_stability"].describe()


(                Country  Year_prev  Year   k  neighbor_stability
 0            Seychelles       1975  1976  12            0.090909
 1      Marshall Islands       1975  1976  12            0.043478
 2              Eswatini       1975  1976  12            0.200000
 3                 Japan       1975  1976  12            0.000000
 4  Hong Kong SAR, China       1975  1976  12            0.000000,
 count    11649.000000
 mean         0.101195
 std          0.168852
 min          0.000000
 25%          0.000000
 50%          0.043478
 75%          0.090909
 max          1.000000
 Name: neighbor_stability, dtype: float64)

Visual 1: Stability distribution per year (how turbulent is the global system?)

In [18]:
fig_stab = px.box(
    stability, x="Year", y="neighbor_stability",
    title=f"Distribution of neighbor-stability (k={k}) — lower = more membership change",
    points="all", height=600
)
fig_stab.update_layout(yaxis=dict(range=[0,1]))
fig_stab.show()
fig_stab.write_html("figures/task2_distribution_of_neighbour_stability.html", include_plotlyjs="cdn", full_html=True)


Visual 2: Country-level stability over time (pick a country or a few)

In [19]:
countries_sample = sorted(stability["Country"].dropna().unique().tolist())[:8]  # change this list as needed

fig_cs = px.line(
    stability[stability["Country"].isin(countries_sample)],
    x="Year", y="neighbor_stability", color="Country",
    markers=True, title=f"Neighbor-stability over time for selected countries (k={k})",
    height=550
)
fig_cs.update_layout(yaxis=dict(range=[0,1]))
fig_cs.show()
fig_cs.write_html("figures/task2_country_level_neighbour_stability.html", include_plotlyjs="cdn", full_html=True)


Interpretation:

- Sharp dips indicate the country changed its local neighborhood ⇒ likely group membership change.

- When several countries dip simultaneously, you’ve found a critical moment worth highlighting in Critical Moment analysis.

# 5. Which attributes define the grouping at each year?

We compute, for every year, how strongly each indicator aligns with the 2D layout by combining the absolute Spearman correlation with x and with y:

alignment
(
𝑎
𝑡
𝑡
𝑟
,
𝑦
𝑒
𝑎
𝑟
)
=
𝜌
(
𝑎
𝑡
𝑡
𝑟
,
𝑥
)
2
+
𝜌
(
𝑎
𝑡
𝑡
𝑟
,
𝑦
)
2
alignment(attr,year)=
ρ(attr,x)
2
+ρ(attr,y)
2
	​


This yields a 0–1 importance score per indicator per year (non-parametric; robust to outliers), showing the mix of defining attributes evolving over time.

In [8]:
from scipy.stats import spearmanr

rows = []
for y, emb in all_embeds.items():
    # Drop rows where both x,y are NaN (shouldn't happen) and require at least 25 valid rows for correlation
    e = emb.copy()
    if len(e) < 25: 
        continue
    for attr in indicators:
        s_attr = pd.to_numeric(e[attr], errors="coerce")
        # Spearman with available pairs only
        mask = s_attr.notna() & e["x"].notna() & e["y"].notna()
        if mask.sum() < 20:
            continue
        rho_x = spearmanr(s_attr[mask], e.loc[mask, "x"]).correlation
        rho_y = spearmanr(s_attr[mask], e.loc[mask, "y"]).correlation
        if np.isnan(rho_x): rho_x = 0.0
        if np.isnan(rho_y): rho_y = 0.0
        align = float(np.sqrt(rho_x**2 + rho_y**2))
        rows.append({"Year": y, "Indicator": attr, "alignment": align})

align_df = pd.DataFrame(rows)
align_df.head()


,Year,Indicator,alignment
0,1975,Inflation,0.129441
1,1975,Credit_to_Private_Sector,0.125449
2,1975,Trade,0.299982
3,1975,GDP,0.411668
4,1975,GDP_Growth,0.324634


Visual 3: Attribute-alignment heatmap (what defines grouping over time)

In [20]:
# Pivot: rows = Indicator, cols = Year
piv = align_df.pivot_table(index="Indicator", columns="Year", values="alignment", aggfunc="mean")
# Reorder indicators by overall mean alignment
order = piv.mean(axis=1).sort_values(ascending=False).index.tolist()
piv = piv.loc[order]

fig_align = px.imshow(
    piv, aspect="auto", origin="lower",
    title="Attribute Alignment with Layout Over Time (Spearman magnitude with x,y)",
    color_continuous_scale="YlGnBu", height=800
)
fig_align.update_layout(xaxis_title="Year", yaxis_title="Indicator")
fig_align.show()
fig_align.write_html("figures/task2_attribute_alignment_heat_map.html", include_plotlyjs="cdn", full_html=True)


Visual 4: Top-k indicators’ alignment as time series

In [21]:
topk = 6
top_inds = (align_df.groupby("Indicator")["alignment"]
            .mean().sort_values(ascending=False).head(topk).index.tolist())

fig_top = px.line(
    align_df[align_df["Indicator"].isin(top_inds)],
    x="Year", y="alignment", color="Indicator",
    markers=True, title=f"Top {topk} defining indicators over time",
    height=550
)
fig_top.update_layout(yaxis=dict(range=[0, align_df["alignment"].max()*1.1]))
fig_top.show()
fig_top.write_html("figures/task2_top_k_indicators.html", include_plotlyjs="cdn", full_html=True)


Interpretation:

- Bright bands in the heatmap show years where a given indicator strongly organizes the map (e.g., Inflation dominates in late 1990s, Credit_to_Private_Sector dominates post-2010, etc.).

- Lines reveal attribute mix shifts—exactly what Temporal Analysis asks for (“combination of attributes defining the grouping changes over time”).

# 6. Drill-down: compare two years side-by-side for visual evidence

A small helper to render two interactive maps for any two selected years with consistent color/size encodings.

In [22]:
def make_embedding_scatter(emb_df, color_col=None, size_col=None, title=None):
    df2 = emb_df.copy()
    # size scaling for visualization only (non-negative)
    if size_col and size_col in df2.columns:
        s = pd.to_numeric(df2[size_col], errors="coerce").clip(lower=0)
        if s.notna().any() and s.max() > 0:
            s_scaled = (s - s.min()) / max(1e-12, s.max() - s.min())
            df2["_size_vis"] = (6 + 24*s_scaled).fillna(6)
            size_use = "_size_vis"
        else:
            size_use = None
    else:
        size_use = None
    hover_cols = ["Country","Year"] + [c for c in indicators][:10]
    fig = px.scatter(
        df2, x="x", y="y", color=color_col if color_col in df2.columns else None,
        size=size_use, hover_data=hover_cols, title=title, height=600
    )
    fig.update_traces(marker=dict(opacity=0.9, line=dict(width=0.5, color="rgba(0,0,0,0.5)")))
    fig.update_layout(xaxis_title="Similarity axis 1", yaxis_title="Similarity axis 2", margin=dict(l=10,r=10,t=60,b=10))
    return fig

year_a, year_b = years_all[0], years_all[-1]  # e.g., earliest vs latest
figA = make_embedding_scatter(all_embeds[year_a], color_col=indicators[0], size_col=indicators[1], title=f"Year {year_a}")
figB = make_embedding_scatter(all_embeds[year_b], color_col=indicators[0], size_col=indicators[1], title=f"Year {year_b}")
figA.show(); 
figA.write_html("figures/task2_embedding_scatter_A.html", include_plotlyjs="cdn", full_html=True)
figB.show()
figB.write_html("figures/task2_embedding_scatter_B.html", include_plotlyjs="cdn", full_html=True)


How it is used in report: place screenshots side-by-side and annotate visible neighborhood shifts for a few exemplar countries (anchor examples).

# 7. “Membership-change watchlist” (auto-detect notable movers)

We’ll rank countries by their average neighbor-stability across all consecutive years; the lowest are your top changers.

In [12]:
watch = (stability.groupby("Country")["neighbor_stability"]
         .mean().sort_values().reset_index(name="avg_stability"))
top_movers = watch.head(15)
top_movers


,Country,avg_stability
0,Turkiye,0.026845
1,Romania,0.028217
2,Greenland,0.029174
3,Sao Tome and Principe,0.029915
4,St. Vincent and the Grenadines,0.030256
5,Tonga,0.032435
6,Pakistan,0.033714
7,Papua New Guinea,0.033999
8,Lesotho,0.034432
9,Central African Republic,0.034867


Visual 5: Focused trajectories for the biggest movers

In [23]:
focus = top_movers["Country"].tolist()[:8]
traj_focus = embeds_df[embeds_df["Country"].isin(focus)].sort_values(["Country","Year"])

fig_mv = go.Figure()
for c, g in traj_focus.groupby("Country"):
    fig_mv.add_trace(go.Scatter(
        x=g["x"], y=g["y"], mode="lines+markers",
        name=c, text=g["Year"], hovertemplate="Country: "+c+"<br>Year: %{text}<extra></extra>"
    ))
fig_mv.update_layout(
    title="Trajectories of largest membership-changers (low neighbor-stability)",
    xaxis_title="Similarity axis 1", yaxis_title="Similarity axis 2", height=700
)
fig_mv.show()
fig_mv.write_html("figures/task2_focused_trajectories.html", include_plotlyjs="cdn", full_html=True)


Use this plot and list in your write-up to provide concrete, visual examples of membership change.

# 8. Quality assurance, performance, and reproducibility

In [14]:
# Sanity checks
assert embeds_df["Country"].isna().sum() == 0, "Country names should be present."
assert embeds_df[["x","y"]].isna().sum().sum() == 0, "Embedding coordinates should be finite."

# Cache to disk (optional) to reuse in your report
OUT_EMB = "cache_embeds.parquet"
OUT_STAB = "neighbor_stability.parquet"
OUT_ALIGN = "attribute_alignment.parquet"
embeds_df.to_parquet(OUT_EMB, index=False)
stability.to_parquet(OUT_STAB, index=False)
align_df.to_parquet(OUT_ALIGN, index=False)
print("Saved:", OUT_EMB, OUT_STAB, OUT_ALIGN)


Saved: cache_embeds.parquet neighbor_stability.parquet attribute_alignment.parquet


The parquete files acan be accessed by any visualization tools

In [15]:
import pandas as pd

# Load the saved parquet files
embeds_df = pd.read_parquet("cache_embeds.parquet")
stability_df = pd.read_parquet("neighbor_stability.parquet")
align_df = pd.read_parquet("attribute_alignment.parquet")

# Check shapes and column names
print("Embeddings file:", embeds_df.shape)
print("Columns:", embeds_df.columns.tolist())

print("\nStability file:", stability_df.shape)
print("Columns:", stability_df.columns.tolist())

print("\nAlignment file:", align_df.shape)
print("Columns:", align_df.columns.tolist())


Embeddings file: (11929, 14)
Columns: ['Country', 'Country Code', 'Year', 'NPLs', 'Inflation', 'Credit_to_Private_Sector', 'Debt', 'Trade', 'GDP', 'GDP_Growth', 'Unemployment', 'x', 'y', '_algo']

Stability file: (11649, 5)
Columns: ['Country', 'Year_prev', 'Year', 'k', 'neighbor_stability']

Alignment file: (347, 3)
Columns: ['Year', 'Indicator', 'alignment']


In [16]:
embeds_df.head(3)
align_df.sample(5)
stability_df.describe()


,Year_prev,Year,k,neighbor_stability
count,11649.000000,11649.000000,11649.0,11649.000000
mean,1999.560735,2000.560735,12.0,0.101195
std,13.587227,13.587227,0.0,0.168852
min,1975.000000,1976.000000,12.0,0.000000
25%,1988.000000,1989.000000,12.0,0.000000
50%,2000.000000,2001.000000,12.0,0.043478
75%,2011.000000,2012.000000,12.0,0.090909
max,2022.000000,2023.000000,12.0,1.000000
